In [1]:
from ForGetEmb.test import get_sel

/home/aidar/projects/mirea_projects/conf-bot/conf-bot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-02-25 16:43:13 nemo_logging:405] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


In [6]:
from faster_whisper import WhisperModel

# Choose a model size: "tiny", "base", "small", "medium", "large"
model_size = "medium" 

# Initialize the model (it will download the model the first time)
# device="cuda" (GPU) or device="cpu" (CPU)
# compute_type="float16" (GPU) or compute_type="int8" (CPU)
model = WhisperModel(model_size, device="cuda", compute_type="float16")



In [7]:

# Transcribe an audio file
# Ensure you have an audio file in a compatible format (e.g., .mp3, .wav, .mp4)
segments, info = model.transcribe("../../data/audio/artur.wav", beam_size=8, word_timestamps=True)

print(f"Detected language: {info.language} with probability {info.language_probability}")

# Print the transcribed text
# for segment in segments:
#     print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

Detected language: ru with probability 0.99267578125


In [8]:
all_words = []

for segment in segments:
    for w in segment.words:
        all_words.append({
            "start": float(w.start),
            "end": float(w.end),
            "word": w.word.strip(),
            # "prob": float(w.probability),
        })

In [9]:
all_words

[{'start': 0.0, 'end': 1.52, 'word': 'Мне'},
 {'start': 1.52, 'end': 1.98, 'word': 'очень'},
 {'start': 1.98, 'end': 2.68, 'word': 'не'},
 {'start': 2.68, 'end': 3.22, 'word': 'понравился'},
 {'start': 3.22, 'end': 3.66, 'word': 'фильм'},
 {'start': 3.66, 'end': 4.24, 'word': '«Грозовой'},
 {'start': 4.24, 'end': 4.7, 'word': 'перевал»,'},
 {'start': 4.7, 'end': 5.0, 'word': 'а'},
 {'start': 5.0, 'end': 5.16, 'word': 'как'},
 {'start': 5.16, 'end': 5.46, 'word': 'тебе,'},
 {'start': 5.48, 'end': 5.92, 'word': 'Даша?'},
 {'start': 6.940000000000004, 'end': 7.78, 'word': 'Средний.'},
 {'start': 8.02, 'end': 8.4, 'word': 'Почему'},
 {'start': 8.4, 'end': 8.8, 'word': 'тебе'},
 {'start': 8.8, 'end': 9.34, 'word': 'не'},
 {'start': 9.34, 'end': 9.94, 'word': 'понравился'},
 {'start': 9.94, 'end': 10.26, 'word': 'фильм'},
 {'start': 10.26, 'end': 10.8, 'word': '«Грозовой'},
 {'start': 10.8, 'end': 11.4, 'word': 'перевал»?'},
 {'start': 11.82, 'end': 12.06, 'word': 'Мне'},
 {'start': 12.06, '

In [2]:
diar_segments = get_sel('../../data/audio/artur.wav')

[NeMo W 2026-02-25 16:43:14 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-02-25 16:43:14 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data

[NeMo I 2026-02-25 16:43:14 nemo_logging:393] PADDING: 16
[NeMo I 2026-02-25 16:43:15 nemo_logging:393] Model EncDecSpeakerLabelModel was successfully restored from /home/aidar/projects/mirea_projects/conf-bot/conf-bot/data/models/titanet_large/speakerverification_en_titanet_large.nemo.
sr: 16000
audio: int16 (2888000,) seconds: 180.5
speech_segments: 33
first speech seg: {'start': 1.328, 'end': 5.968, 'flag': 1, 'mean_prob': 0.8379805839874528, 'n_frames': 264}
windows: 202
first window: (1.328, 3.3280000000000003) dur: 2.0


In [ ]:
def merge_consecutive_by_label(segments):
    if not segments:
        return []

    merged = []
    cur_start, cur_end, cur_label = segments[0]

    for start, end, label in segments[1:]:
        if label == cur_label:
            cur_end = max(cur_end, end)  # тянем конец до последнего куска
        else:
            merged.append((cur_start, cur_end, cur_label))
            cur_start, cur_end, cur_label = start, end, label

    merged.append((cur_start, cur_end, cur_label))
    return merged


segments_window_level = merge_consecutive_by_label(diar_segments)


In [14]:
segments_window_level

[(1.328, 5.828, 0),
 (6.944, 10.943999999999999, 1),
 (11.84, 28.752, 0),
 (29.552, 32.552, 1),
 (31.052, 33.052, -1),
 (31.552, 58.628, 1),
 (59.52, 91.424, 0),
 (93.12, 105.492, 1),
 (106.448, 108.448, -1),
 (109.072, 122.572, 1),
 (124.912, 159.564, 0),
 (160.496, 166.856, 1),
 (165.356, 167.356, -1),
 (165.856, 177.94, 1)]

In [21]:
def overlap(a_start, a_end, b_start, b_end):
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))


def attach_text_to_diar(diar_segments, words):
    results = []

    for d_start, d_end, spk in diar_segments:
        segment_words = []

        for w in words:
            if overlap(w["start"], w["end"], d_start, d_end) > 0:
                segment_words.append(w)

        if not segment_words:
            continue

        text = " ".join(w["word"] for w in segment_words)

        new_start = min(w["start"] for w in segment_words)
        new_end = max(w["end"] for w in segment_words)

        results.append({
            "start": new_start,
            "end": new_end,
            "speaker": spk,
            "text": text
        })

    return results

In [23]:
aligned_words = attach_text_to_diar(segments_window_level, all_words)

In [24]:
for w in aligned_words[:20]:
    print(w)

{'start': 0.0, 'end': 5.92, 'speaker': 0, 'text': 'Мне очень не понравился фильм «Грозовой перевал», а как тебе, Даша?'}
{'start': 6.940000000000004, 'end': 11.4, 'speaker': 1, 'text': 'Средний. Почему тебе не понравился фильм «Грозовой перевал»?'}
{'start': 11.82, 'end': 28.88, 'speaker': 0, 'text': 'Мне он не понравился, потому что сюжет в нём был не очень для меня. Мне не особо близко... близок сюжет, в котором всё разворачивается как -то пошло. Как по мне, это было очень пошло.'}
{'start': 28.88, 'end': 32.5, 'speaker': 1, 'text': 'Какой ты скромный, Артур.'}
{'start': 31.34, 'end': 32.5, 'speaker': -1, 'text': 'Артур.'}
{'start': 31.34, 'end': 58.92, 'speaker': 1, 'text': 'Артур. Мне понравился, как снят фильм. Мне не нравится в целом история больных вот этих вот взаимоотношений, которые там описаны. Но когда я читала книгу, мне и книга тоже не нравилась, потому что я ничего не понимала, что там происходит. А ты, после просмотра фильма, хотела бы прочитать книгу, чтобы, может быть